# Keras Framework and MNIST Classification

This notebook covers:
- Loading and preparing MNIST dataset
- Building neural networks with Keras Sequential API
- Dense and Flatten layers
- Model compilation and training
- Interpreting training history and metrics

In [ ]:
# Import libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## Loading and Preparing MNIST

MNIST (Modified National Institute of Standards and Technology) is a classic dataset of handwritten digits (0-9). It contains:
- 60,000 training images
- 10,000 test images
- 28×28 pixel grayscale images
- 10 classes (digits 0-9)

In [ ]:
# Load MNIST
print("Loading MNIST dataset...")
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print(f"Training images shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test images shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Pixel value range: [{X_train.min()}, {X_train.max()}]")
print(f"Unique labels: {np.unique(y_train)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

print("\nSample images displayed above")

In [ ]:
# Normalize to [0, 1] - important for training stability
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"After normalization:")
print(f"Training data range: [{X_train.min()}, {X_train.max()}]")
print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

## Building a Neural Network Model

The Sequential API stacks layers in order. For images, we typically:
1. **Flatten** the 2D image to 1D vector (28×28 → 784)
2. Add **Dense** (fully connected) hidden layers with activation
3. Add **Dense** output layer with softmax for classification

In [ ]:
# Build the model
model = tf.keras.Sequential([
    # Input shape: 28×28 images
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    
    # Hidden layer 1: 256 neurons, ReLU activation
    tf.keras.layers.Dense(256, activation='relu'),
    
    # Hidden layer 2: 128 neurons, ReLU activation
    tf.keras.layers.Dense(128, activation='relu'),
    
    # Output layer: 10 neurons (one per digit), softmax for probabilities
    tf.keras.layers.Dense(10, activation='softmax')
])

# Display model architecture
print("Model Architecture:")
model.summary()

## Compiling the Model

Compilation specifies:
- **Optimizer**: Updates weights (Adam is adaptive, works well in practice)
- **Loss function**: Measures prediction error (sparse_categorical_crossentropy for integer labels)
- **Metrics**: Tracks performance (accuracy)

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Training the Model

Training parameters:
- **epochs**: How many times to iterate over full training data
- **batch_size**: How many samples to process before updating weights
- **validation_split**: Fraction of training data to use for validation

In [ ]:
# Train the model
print("Training model on MNIST...")
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

print("\nTraining completed!")

## Evaluating Model Performance

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")

# Also evaluate on training set to detect overfitting
train_loss, train_accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f"\nTrain Results:")
print(f"  Loss: {train_loss:.4f}")
print(f"  Accuracy: {train_accuracy:.4f}")

## Making Predictions

In [ ]:
# Get predictions for test set
predictions = model.predict(X_test[:10])

print("Predictions for first 10 test images:")
print("\nImage | Predicted | Actual | Correct?")
print("------|-----------|--------|----------")
for i in range(10):
    pred_class = np.argmax(predictions[i])
    actual_class = y_test[i]
    correct = "✓" if pred_class == actual_class else "✗"
    confidence = np.max(predictions[i])
    print(f"  {i:2d}  |    {pred_class}     |   {actual_class}    |  {correct} ({confidence:.2%})")

## Visualizing Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', marker='o', markersize=4)
axes[0].plot(history.history['val_loss'], label='Validation Loss', marker='s', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history.history['accuracy'], label='Training Accuracy', marker='o', markersize=4)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations from training history:")
print("- Training and validation curves should improve together")
print("- If val_loss rises while train_loss falls → overfitting (memorization)")
print("- If both still improving → could train longer")
print("- Smooth curves indicate stable training")